# Machine Learning Classification of Bacterial Vaginosis Using African Cohort Data
## Willy KAYONDO
## Reg.No. 2025/HD07/25992U
## MSB7215
### 08 June 2026

# Introduction

# Objective

In [1]:
%%bash
ls "/mnt/c/MsC_Bioinformatics/Machine_Learning/bv_project/rwanda"

srep14174-s2.xls
srep14174-s3.xls
srep14174-s5.xls


In [7]:
%%bash
pip install xlrd


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [14]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = "/mnt/c/MsC_Bioinformatics/Machine_Learning/bv_project/rwanda"

# ── METADATA (s2) ─────────────────────────────────────────────────
meta_raw = pd.read_excel(f"{DATA_DIR}/srep14174-s2.xls", header=1)
meta_raw = meta_raw.rename(columns={meta_raw.columns[0]: "Patient_ID"})
meta_raw = meta_raw[pd.to_numeric(meta_raw["Patient_ID"],
                    errors='coerce').notna()].copy()
meta_raw["Patient_ID"] = meta_raw["Patient_ID"].astype(int)
meta = meta_raw.set_index("Patient_ID")
print(f"✓ Meta loaded: {meta.shape}")
print(f"  Index (first 5): {meta.index.tolist()[:5]}")

# ── GC-MS METABOLOMICS (s3) ───────────────────────────────────────
s3_raw = pd.read_excel(f"{DATA_DIR}/srep14174-s3.xls", header=2, index_col=0)

numeric_cols_s3 = []
for c in s3_raw.columns:
    try:
        if float(str(c)) > 1:
            numeric_cols_s3.append(c)
    except:
        pass

s3_clean = s3_raw[numeric_cols_s3].copy()
non_taxa = ['Nugent Status', 'Pregnant', 'diversity', 'Nugent status']
s3_clean = s3_clean[~s3_clean.index.isin(non_taxa)]
s3_clean = s3_clean[s3_clean.index.notna()]
gcms = s3_clean.T.copy()
gcms.index = pd.to_numeric(gcms.index, errors='coerce').astype(int)
print(f"\n✓ GC-MS metabolomics: {gcms.shape}")
print(f"  Sample IDs (first 5): {gcms.index.tolist()[:5]}")

# ── 16S OTU TABLE (s5) ────────────────────────────────────────────
# Row 0 = title, Row 1 = headers (Taxa, sample IDs), Row 2+ = taxa
s5_raw = pd.read_excel(f"{DATA_DIR}/srep14174-s5.xls", header=1, index_col=0)

# Keep only numeric sample ID columns (patient IDs)
numeric_cols_s5 = []
for c in s5_raw.columns:
    try:
        val = float(str(c))
        if val >= 1:
            numeric_cols_s5.append(c)
    except:
        pass

s5_clean = s5_raw[numeric_cols_s5].copy()
s5_clean = s5_clean[s5_clean.index.notna()]
s5_clean = s5_clean[s5_clean.index != 'Taxa']

# Transpose: samples as rows, taxa as columns
otu = s5_clean.T.copy()
otu.index = pd.to_numeric(otu.index, errors='coerce').astype(int)

# Shorten taxa names (take last part after final semicolon)
otu.columns = [str(c).split(';')[-1].strip() for c in otu.columns]

print(f"\n✓ 16S OTU table: {otu.shape}")
print(f"  Sample IDs (first 5): {otu.index.tolist()[:5]}")
print(f"  Taxa (first 3): {otu.columns.tolist()[:3]}")

# ── BV LABEL ──────────────────────────────────────────────────────
meta["BV_status"] = (meta["Nugent"] >= 7).astype(int)

# ── ALIGN ALL THREE ON COMMON PATIENT IDs ─────────────────────────
common_ids = meta.index.intersection(gcms.index).intersection(otu.index)
meta  = meta.loc[common_ids]
gcms  = gcms.loc[common_ids]
otu   = otu.loc[common_ids]

# Clean NaN
gcms = gcms.dropna(axis=1, how='all').fillna(gcms.median())
otu  = otu.dropna(axis=1,  how='all').fillna(0)

# ── COMBINED FEATURE MATRIX ───────────────────────────────────────
metab = pd.concat([gcms, otu], axis=1)
y     = meta["BV_status"].values

print("\n" + "=" * 45)
print(f"  Common samples        : {len(common_ids)}")
print(f"  Meta shape            : {meta.shape}")
print(f"  GC-MS shape           : {gcms.shape}")
print(f"  OTU shape             : {otu.shape}")
print(f"  Combined metab shape  : {metab.shape}")
print(f"\n  BV  (1)    : {meta['BV_status'].sum()} samples")
print(f"  Non-BV (0) : {(meta['BV_status']==0).sum()} samples")
print("=" * 45)

✓ Meta loaded: (131, 25)
  Index (first 5): [1, 3, 4, 5, 6]

✓ GC-MS metabolomics: (130, 128)
  Sample IDs (first 5): [3, 15, 16, 25, 30]

✓ 16S OTU table: (131, 51)
  Sample IDs (first 5): [1, 3, 4, 5, 6]
  Taxa (first 3): ['Lactobacillus_iners', 'Lactobacillus_crispatus', 'Lactobacillus']

  Common samples        : 130
  Meta shape            : (130, 26)
  GC-MS shape           : (130, 128)
  OTU shape             : (130, 51)
  Combined metab shape  : (130, 179)

  BV  (1)    : 23 samples
  Non-BV (0) : 107 samples


In [ ]:
# CHECKPOINT — checking variables
print("META:")
print(f"  Shape : {meta.shape}")
print(f"  Index (first 5): {meta.index.tolist()[:5]}")
print(f"  BV_status counts:\n{meta['BV_status'].value_counts()}")

print("\nGC-MS (gcms):")
print(f"  Shape : {gcms.shape}")
print(f"  Index (first 5): {gcms.index.tolist()[:5]}")

print("\nOTU:")
print(f"  Shape : {otu.shape}")
print(f"  Index (first 5): {otu.index.tolist()[:5]}")

print("\nCOMBINED (metab):")
print(f"  Shape : {metab.shape}")

print(f"\ny (label vector): {y[:10]}")
print(f"y shape: {y.shape}")

META:
  Shape : (130, 26)
  Index (first 5): [3, 4, 5, 6, 7]
  BV_status counts:
BV_status
0    107
1     23
Name: count, dtype: int64

GC-MS (gcms):
  Shape : (130, 128)
  Index (first 5): [3, 4, 5, 6, 7]

OTU:
  Shape : (130, 51)
  Index (first 5): [3, 4, 5, 6, 7]

COMBINED (metab):
  Shape : (130, 179)

y (label vector): [1 0 0 0 0 0 0 0 0 0]
y shape: (130,)


In [16]:
# FEATURE ENGINEERING AND DEFINING ALGORITHMS
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# ── SCALE FEATURES ────────────────────────────────────────────────
scaler = StandardScaler()

X_gcms  = scaler.fit_transform(gcms.values)   # metabolomics only
X_otu   = scaler.fit_transform(otu.values)    # 16S OTU only
X_multi = scaler.fit_transform(metab.values)  # combined (179 features)

print("Feature matrices ready:")
print(f"  GC-MS (metabolomics) : {X_gcms.shape}")
print(f"  OTU (16S microbiome) : {X_otu.shape}")
print(f"  Combined (multi-omics): {X_multi.shape}")
print(f"  Labels               : {y.shape} | BV={y.sum()} | Non-BV={len(y)-y.sum()}")

# ── DEFINE ALGORITHMS ─────────────────────────────────────────────
# class_weight='balanced' handles the 107:23 imbalance automatically
scale_pos = (y == 0).sum() / (y == 1).sum()  # = 107/23 ≈ 4.65

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=42,
        class_weight='balanced'),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        class_weight='balanced'),

    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42,
        class_weight='balanced'),

    "SVM": SVC(
        kernel='rbf', probability=True,
        random_state=42,
        class_weight='balanced'),

    "KNN": KNeighborsClassifier(
        n_neighbors=5),

    "XGBoost": XGBClassifier(
        n_estimators=200, random_state=42,
        use_label_encoder=False,
        eval_metric='logloss',
        scale_pos_weight=scale_pos)
}

print(f"\n✓ {len(models)} algorithms defined:")
for name in models:
    print(f"  • {name}")

Feature matrices ready:
  GC-MS (metabolomics) : (130, 128)
  OTU (16S microbiome) : (130, 51)
  Combined (multi-omics): (130, 179)
  Labels               : (130,) | BV=23 | Non-BV=107

✓ 6 algorithms defined:
  • Logistic Regression
  • Decision Tree
  • Random Forest
  • SVM
  • KNN
  • XGBoost


In [17]:
# EVALUATE ALL MODELS ACROSS 3 FEATURE SETS
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_feature_set(X, feature_set_name):
    """Evaluate all models on a given feature set."""
    res = []
    print(f"\n{'='*55}")
    print(f"  Feature set: {feature_set_name}")
    print(f"{'='*55}")
    for name, model in models.items():
        y_pred  = cross_val_predict(model, X, y, cv=cv,
                                    method='predict')
        y_proba = cross_val_predict(model, X, y, cv=cv,
                                    method='predict_proba')[:, 1]
        row = {
            "Model"    : name,
            "Accuracy" : round(accuracy_score(y, y_pred),            3),
            "Precision": round(precision_score(y, y_pred,
                                zero_division=0),                     3),
            "Recall"   : round(recall_score(y, y_pred,
                                zero_division=0),                     3),
            "F1-Score" : round(f1_score(y, y_pred,
                                zero_division=0),                     3),
            "ROC-AUC"  : round(roc_auc_score(y, y_proba),            3)
        }
        res.append(row)
        print(f"\n  {name}")
        print(f"    Accuracy={row['Accuracy']}  "
              f"Precision={row['Precision']}  "
              f"Recall={row['Recall']}  "
              f"F1={row['F1-Score']}  "
              f"AUC={row['ROC-AUC']}")
    return pd.DataFrame(res).sort_values("ROC-AUC", ascending=False)

# ── EVALUATE ALL THREE FEATURE SETS ───────────────────────────────
results_gcms  = evaluate_feature_set(X_gcms,  "GC-MS Metabolomics (128 features)")
results_otu   = evaluate_feature_set(X_otu,   "16S OTU Microbiome (51 features)")
results_multi = evaluate_feature_set(X_multi, "Multi-omics Combined (179 features)")

# ── PRINT SUMMARY TABLES ──────────────────────────────────────────
print("\n\n=== GC-MS METABOLOMICS ===")
print(results_gcms.to_string(index=False))

print("\n\n=== 16S OTU MICROBIOME ===")
print(results_otu.to_string(index=False))

print("\n\n=== MULTI-OMICS COMBINED ===")
print(results_multi.to_string(index=False))

# ── COMPARISON BAR CHART ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
datasets = [
    (results_gcms,  "GC-MS Metabolomics"),
    (results_otu,   "16S OTU Microbiome"),
    (results_multi, "Multi-omics Combined")
]

for ax, (res_df, title) in zip(axes, datasets):
    x     = np.arange(len(res_df))
    width = 0.15
    for i, metric in enumerate(metrics):
        ax.bar(x + i*width, res_df[metric], width, label=metric)
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(res_df["Model"], rotation=30,
                       ha='right', fontsize=8)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle(
    "BV vs Non-BV Classification — Rwanda Cohort (n=130)\n"
    "Model Comparison Across Feature Sets",
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/model_comparison.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Comparison plot saved → model_comparison.png")

# ── CONFUSION MATRIX for best model on best feature set ───────────
best_name   = results_multi.iloc[0]["Model"]
best_model  = models[best_name]
y_pred_best = cross_val_predict(best_model, X_multi, y, cv=cv)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y, y_pred_best),
    display_labels=["Non-BV (0)", "BV (1)"]
).plot(ax=ax, cmap='Blues')
ax.set_title(f"Confusion Matrix\n{best_name} — Multi-omics",
             fontweight='bold')
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/confusion_matrix.png",
            dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Confusion matrix saved → confusion_matrix.png")
print(f"✓ Best performing model: {best_name}")


  Feature set: GC-MS Metabolomics (128 features)

  Logistic Regression
    Accuracy=0.838  Precision=0.545  Recall=0.522  F1=0.533  AUC=0.864

  Decision Tree
    Accuracy=0.831  Precision=0.522  Recall=0.522  F1=0.522  AUC=0.709

  Random Forest
    Accuracy=0.877  Precision=0.889  Recall=0.348  F1=0.5  AUC=0.889

  SVM
    Accuracy=0.885  Precision=0.786  Recall=0.478  F1=0.595  AUC=0.918

  KNN
    Accuracy=0.885  Precision=0.786  Recall=0.478  F1=0.595  AUC=0.795

  XGBoost
    Accuracy=0.892  Precision=0.765  Recall=0.565  F1=0.65  AUC=0.889

  Feature set: 16S OTU Microbiome (51 features)

  Logistic Regression
    Accuracy=0.869  Precision=0.607  Recall=0.739  F1=0.667  AUC=0.911

  Decision Tree
    Accuracy=0.846  Precision=0.565  Recall=0.565  F1=0.565  AUC=0.736

  Random Forest
    Accuracy=0.854  Precision=0.625  Recall=0.435  F1=0.513  AUC=0.876

  SVM
    Accuracy=0.869  Precision=0.615  Recall=0.696  F1=0.653  AUC=0.928

  KNN
    Accuracy=0.862  Precision=0.647  Reca

In [21]:
# OPTMIZING THE MODEL
from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE

cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── PRE-OPTIMIZATION BASELINE ─────────────────────────────────────
best_name  = results_multi.iloc[0]["Model"]
best_model = models[best_name]

y_pred_pre  = cross_val_predict(best_model, X_multi, y, cv=cv_tune)
y_proba_pre = cross_val_predict(best_model, X_multi, y, cv=cv_tune,
                                method='predict_proba')[:, 1]

print("=" * 50)
print("  PRE-OPTIMIZATION BASELINE")
print(f"  Model     : {best_name}")
print(f"  Feature set: Multi-omics (179 features)")
print(f"  Accuracy  : {accuracy_score(y, y_pred_pre):.3f}")
print(f"  Precision : {precision_score(y, y_pred_pre, zero_division=0):.3f}")
print(f"  Recall    : {recall_score(y, y_pred_pre, zero_division=0):.3f}")
print(f"  F1-Score  : {f1_score(y, y_pred_pre, zero_division=0):.3f}")
print(f"  ROC-AUC   : {roc_auc_score(y, y_proba_pre):.3f}")
print("=" * 50)

# ── OPTIMIZATION 1: SMOTE (k=3 safe for small BV class n=23) ──────
print("\n--- Optimization 1: SMOTE Class Balancing ---")
smote        = SMOTE(random_state=42, k_neighbors=3)
X_res, y_res = smote.fit_resample(X_multi, y)
print(f"  Before: Non-BV={( y==0).sum()}  BV={( y==1).sum()}")
print(f"  After : Non-BV={(y_res==0).sum()}  BV={(y_res==1).sum()}")

# ── OPTIMIZATION 2: Feature Selection ────────────────────────────
print("\n--- Optimization 2: SelectKBest (top 40 of 179) ---")
selector   = SelectKBest(f_classif, k=40)
X_selected = selector.fit_transform(X_res, y_res)

all_feature_names = list(gcms.columns) + list(otu.columns)
sel_idx   = selector.get_support(indices=True)
sel_names = [all_feature_names[i] for i in sel_idx]

print(f"  Features: 179 → {X_selected.shape[1]}")
print(f"  Top 5 selected: {sel_names[:5]}")

# ── OPTIMIZATION 3: Hyperparameter Tuning ────────────────────────
print("\n--- Optimization 3: RandomizedSearchCV (20 iterations each) ---")

param_grids = {
    "Random Forest": {
        "model": RandomForestClassifier(
            random_state=42, class_weight='balanced'),
        "params": {
            "n_estimators"     : [100, 200, 300],
            "max_depth"        : [None, 5, 10, 15],
            "min_samples_split": [2, 5, 10],
            "max_features"     : ["sqrt", "log2"]
        }
    },
    "XGBoost": {
        "model": XGBClassifier(
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss'),
        "params": {
            "n_estimators" : [100, 200, 300],
            "max_depth"    : [3, 5, 7],
            "learning_rate": [0.01, 0.05, 0.1, 0.2],
            "subsample"    : [0.7, 0.8, 1.0]
        }
    },
    "SVM": {
        "model": SVC(
            probability=True,
            class_weight='balanced',
            random_state=42),
        "params": {
            "C"     : [0.1, 1, 10, 100],
            "gamma" : ["scale", "auto", 0.01, 0.001],
            "kernel": ["rbf", "linear"]
        }
    },
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=1000, random_state=42,
            class_weight='balanced'),
        "params": {
            "C"      : [0.01, 0.1, 1, 10, 100],
            "penalty": ["l1", "l2"],
            "solver" : ["liblinear", "saga"]
        }
    }
}

tuning_results = {}
post_results   = []

for model_name, config in param_grids.items():
    print(f"\n  Tuning {model_name}...")
    search = RandomizedSearchCV(
        estimator           = config["model"],
        param_distributions = config["params"],
        n_iter              = 20,
        scoring             = "roc_auc",
        cv                  = cv_tune,
        random_state        = 42,
        n_jobs              = -1,
        verbose             = 0
    )
    search.fit(X_selected, y_res)
    tuning_results[model_name] = search

    best  = search.best_estimator_
    y_p   = cross_val_predict(best, X_selected, y_res, cv=cv_tune)
    y_pr  = cross_val_predict(best, X_selected, y_res, cv=cv_tune,
                               method='predict_proba')[:, 1]

    post_results.append({
        "Model"    : model_name,
        "Accuracy" : round(accuracy_score(y_res, y_p),           3),
        "Precision": round(precision_score(y_res, y_p,
                           zero_division=0),                      3),
        "Recall"   : round(recall_score(y_res, y_p,
                           zero_division=0),                      3),
        "F1-Score" : round(f1_score(y_res, y_p,
                           zero_division=0),                      3),
        "ROC-AUC"  : round(roc_auc_score(y_res, y_pr),           3)
    })
    print(f"    Best AUC   : {search.best_score_:.3f}")
    print(f"    Best params: {search.best_params_}")

post_df = pd.DataFrame(post_results).sort_values("ROC-AUC", ascending=False)

print("\n\n=== POST-OPTIMIZATION RESULTS ===")
print(post_df[["Model","Accuracy","Precision",
               "Recall","F1-Score","ROC-AUC"]].to_string(index=False))

# ── BEFORE vs AFTER COMPARISON PLOT ──────────────────────────────
before_map = {r["Model"]: r["ROC-AUC"]
              for r in results_multi.to_dict('records')
              if r["Model"] in param_grids}
after_map  = {r["Model"]: r["ROC-AUC"] for r in post_results}

model_names = list(param_grids.keys())
before_vals = [before_map.get(m, 0) for m in model_names]
after_vals  = [after_map.get(m, 0)  for m in model_names]

x     = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, before_vals, width,
               label='Before Optimization',
               color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, after_vals, width,
               label='After Optimization',
               color='darkorange', alpha=0.85)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.3f}",
            ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.3f}",
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel("ROC-AUC Score", fontsize=11)
ax.set_ylim(0, 1.15)
ax.axhline(0.8, color='red', linestyle='--',
           alpha=0.5, label='0.8 threshold')
ax.set_title(
    "BV Classification — ROC-AUC Before vs After Optimization\n"
    "Rwanda Cohort (n=130) | SMOTE + Feature Selection + Hyperparameter Tuning",
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/optimization_comparison.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Optimization comparison plot saved")

# ── FEATURE IMPORTANCE (best tuned model) ────────────────────────
best_tuned_name = post_df.iloc[0]["Model"]
best_tuned      = tuning_results[best_tuned_name].best_estimator_
best_tuned.fit(X_selected, y_res)

if hasattr(best_tuned, 'feature_importances_'):
    feat_df = pd.DataFrame({
        "Feature"   : sel_names,
        "Importance": best_tuned.feature_importances_
    }).sort_values("Importance", ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(feat_df["Feature"][::-1],
            feat_df["Importance"][::-1],
            color='teal', alpha=0.85)
    ax.set_xlabel("Feature Importance", fontsize=11)
    ax.set_title(
        f"Top 15 Features Driving BV Classification\n"
        f"{best_tuned_name} — Rwanda Cohort",
        fontweight='bold', fontsize=11
    )
    plt.tight_layout()
    plt.savefig(f"{DATA_DIR}/feature_importance.png",
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Top 5 features driving BV classification:")
    print(feat_df[["Feature","Importance"]].head(5).to_string(index=False))

elif hasattr(best_tuned, 'coef_'):
    # For Logistic Regression
    coef_df = pd.DataFrame({
        "Feature"    : sel_names,
        "Coefficient": np.abs(best_tuned.coef_[0])
    }).sort_values("Coefficient", ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(coef_df["Feature"][::-1],
            coef_df["Coefficient"][::-1],
            color='teal', alpha=0.85)
    ax.set_xlabel("|Coefficient|", fontsize=11)
    ax.set_title(
        f"Top 15 Features — {best_tuned_name}\nRwanda Cohort",
        fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(f"{DATA_DIR}/feature_importance.png",
                dpi=150, bbox_inches='tight')
    plt.show()

print(f"\n{'='*50}")
print(f"  OPTIMIZATION COMPLETE")
print(f"  Best model: {best_tuned_name}")
print(f"  Best AUC  : {post_df.iloc[0]['ROC-AUC']}")
print(f"  Plots saved to: {DATA_DIR}")
print(f"{'='*50}")

  PRE-OPTIMIZATION BASELINE
  Model     : SVM
  Feature set: Multi-omics (179 features)
  Accuracy  : 0.892
  Precision : 0.846
  Recall    : 0.478
  F1-Score  : 0.611
  ROC-AUC   : 0.931

--- Optimization 1: SMOTE Class Balancing ---
  Before: Non-BV=107  BV=23
  After : Non-BV=107  BV=107

--- Optimization 2: SelectKBest (top 40 of 179) ---
  Features: 179 → 40
  Top 5 selected: ['tyrosine', 'N-acetyl-lysine', 'ornithine', 'unknown sugar 9', 'unknown sugar 1']

--- Optimization 3: RandomizedSearchCV (20 iterations each) ---

  Tuning Random Forest...
    Best AUC   : 0.991
    Best params: {'n_estimators': 100, 'min_samples_split': 2, 'max_features': 'log2', 'max_depth': None}

  Tuning XGBoost...
    Best AUC   : 0.993
    Best params: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}

  Tuning SVM...
    Best AUC   : 0.998
    Best params: {'kernel': 'rbf', 'gamma': 'auto', 'C': 100}

  Tuning Logistic Regression...
    Best AUC   : 0.983
    Best params

In [ ]:
#!pip install imbalanced-learn

  Using cached imbalanced_learn-0.14.2-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.6-py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn][imbalanced-learn]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [24]:
# GENERATE FEATURE IMPORTANCE PLOT
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Use Random Forest (which always has feature_importances_)
# Refit on selected features
from sklearn.ensemble import RandomForestClassifier

rf_for_importance = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)
rf_for_importance.fit(X_selected, y_res)

# Build importance dataframe
feat_df = pd.DataFrame({
    "Feature"   : sel_names,
    "Importance": rf_for_importance.feature_importances_
}).sort_values("Importance", ascending=False).head(15)

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(feat_df["Feature"][::-1],
        feat_df["Importance"][::-1],
        color='teal', alpha=0.85)
ax.set_xlabel("Feature Importance", fontsize=11)
ax.set_title(
    "Top 15 Features Driving BV Classification\n"
    "Random Forest — Rwanda Cohort (n=130)",
    fontweight='bold', fontsize=11
)
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/feature_importance.png",
            dpi=150, bbox_inches='tight')
plt.show()

print("✓ feature_importance.png saved")
print("\nTop 5 features:")
print(feat_df[["Feature","Importance"]].head(5).to_string(index=False))

✓ feature_importance.png saved

Top 5 features:
                       Feature  Importance
                   Gardnerella    0.117884
                     Atopobium    0.115730
                           GHB    0.112054
                    Prevotella    0.069581
Parvimonas(Peptostreptococcus)    0.050776


In [25]:
# ── FINAL SUMMARY TABLE ───────────────────────────────────────────
print("=" * 60)
print("  DISSERTATION RESULTS SUMMARY")
print("  Project: BV Classification — Rwanda Cohort (n=130)")
print("  Features: GC-MS Metabolomics + 16S OTU (179 combined)")
print("=" * 60)

print("\n📊 STEP 5 — Best Model Per Feature Set (Before Optimization):")
summary = {
    "GC-MS Metabolomics (128)"  : results_gcms.iloc[0],
    "16S OTU Microbiome (51)"   : results_otu.iloc[0],
    "Multi-omics Combined (179)": results_multi.iloc[0]
}
for feat_set, row in summary.items():
    print(f"\n  {feat_set}")
    print(f"    Best model : {row['Model']}")
    print(f"    Accuracy   : {row['Accuracy']}")
    print(f"    F1-Score   : {row['F1-Score']}")
    print(f"    ROC-AUC    : {row['ROC-AUC']}")

print("\n⚙️  STEP 6 — After Optimization (SMOTE + SelectKBest + Tuning):")
print(post_df[["Model","Accuracy","Precision",
               "Recall","F1-Score","ROC-AUC"]].to_string(index=False))

print(f"\n🏆 BEST OVERALL MODEL: {post_df.iloc[0]['Model']}")
print(f"   ROC-AUC : {post_df.iloc[0]['ROC-AUC']}")
print(f"   F1-Score: {post_df.iloc[0]['F1-Score']}")

print("\n📁 Saved outputs:")
import os
for f in ["model_comparison.png", "confusion_matrix.png",
          "optimization_comparison.png", "feature_importance.png"]:
    path   = f"{DATA_DIR}/{f}"
    exists = "✓" if os.path.exists(path) else "✗ missing"
    print(f"   {exists}  {f}")

print("\n" + "=" * 60)
print("  Steps 3-6 complete.")
print("=" * 60)

  DISSERTATION RESULTS SUMMARY
  Project: BV Classification — Rwanda Cohort (n=130)
  Features: GC-MS Metabolomics + 16S OTU (179 combined)

📊 STEP 5 — Best Model Per Feature Set (Before Optimization):

  GC-MS Metabolomics (128)
    Best model : SVM
    Accuracy   : 0.885
    F1-Score   : 0.595
    ROC-AUC    : 0.918

  16S OTU Microbiome (51)
    Best model : SVM
    Accuracy   : 0.869
    F1-Score   : 0.653
    ROC-AUC    : 0.928

  Multi-omics Combined (179)
    Best model : SVM
    Accuracy   : 0.892
    F1-Score   : 0.611
    ROC-AUC    : 0.931

⚙️  STEP 6 — After Optimization (SMOTE + SelectKBest + Tuning):
              Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
                SVM     0.963      0.938   0.991     0.964    0.998
      Random Forest     0.949      0.944   0.953     0.949    0.990
            XGBoost     0.953      0.937   0.972     0.954    0.990
Logistic Regression     0.935      0.912   0.963     0.936    0.980

🏆 BEST OVERALL MODEL: SVM
   ROC-AUC 

In [26]:
# Extract Table 1 values from Cell 7 results
print("=== TABLE 1: BASELINE MULTI-OMICS RESULTS ===")
print("Copy these into your LaTeX Table 1\n")

for _, row in results_multi.iterrows():
    print(f"{row['Model']:<22} & {row['Accuracy']} & {row['Precision']} "
          f"& {row['Recall']} & {row['F1-Score']} & {row['ROC-AUC']} \\\\")

=== TABLE 1: BASELINE MULTI-OMICS RESULTS ===
Copy these into your LaTeX Table 1

SVM                    & 0.892 & 0.846 & 0.478 & 0.611 & 0.931 \\
Logistic Regression    & 0.9 & 0.75 & 0.652 & 0.698 & 0.91 \\
XGBoost                & 0.885 & 0.7 & 0.609 & 0.651 & 0.899 \\
Random Forest          & 0.885 & 0.833 & 0.435 & 0.571 & 0.889 \\
KNN                    & 0.885 & 0.833 & 0.435 & 0.571 & 0.84 \\
Decision Tree          & 0.815 & 0.476 & 0.435 & 0.455 & 0.666 \\


In [27]:
# Extract Table 2 values from Cell 8 results
print("=== TABLE 2: POST-OPTIMISATION RESULTS ===")
print("Copy these into your LaTeX Table 2\n")

for _, row in post_df.iterrows():
    print(f"{row['Model']:<22} & {row['Accuracy']} & {row['Precision']} "
          f"& {row['Recall']} & {row['F1-Score']} & {row['ROC-AUC']} \\\\")

=== TABLE 2: POST-OPTIMISATION RESULTS ===
Copy these into your LaTeX Table 2

SVM                    & 0.963 & 0.938 & 0.991 & 0.964 & 0.998 \\
Random Forest          & 0.949 & 0.944 & 0.953 & 0.949 & 0.99 \\
XGBoost                & 0.953 & 0.937 & 0.972 & 0.954 & 0.99 \\
Logistic Regression    & 0.935 & 0.912 & 0.963 & 0.936 & 0.98 \\
